In [18]:
!pip install -q sentence-transformers

In [19]:
pip install -U bitsandbytes -q

Note: you may need to restart the kernel to use updated packages.


In [20]:
!pip install -q gensim

In [21]:
!pip install accelerate -q

In [22]:
!pip install datasets -q

In [23]:
ALPHA = 0.35

FULL_ADAPTIVE = {
    "triviaqa": {"ECR": 92.2, "SSC": 86.9, "APSS": 0.68, "API": 11.6, "Abstain": 40.6},
    "webq":     {"ECR": 81.8, "SSC": 68.1, "APSS": 1.57, "API": 12.6, "Abstain": 43.0},
}

In [24]:
import random
import numpy as np
import torch

In [25]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

2.11.0+cu130
0.26.0+cu130


In [26]:
from model import load_sentence_encoder, load_model, generate_response, generate_batch
from scoring import clean_response, compute_stability, is_uncertainty_response, plateau_stop, responses_are_substantive
from data import load_triviaqa, load_mmlu, load_webquestions
from evaluate import evaluate, run_full_evaluation, print_comparison_table
from calibration import compute_qhat, check_coverage, run_calibration
from sampler import adaptive_sample, check_coverage, build_prediction_set
from helpers import _assign_split, _print_split_counts, filter_split, clear_embed_cache, embed_cached
from scoring import clean_response, compute_stability, plateau_stop, is_uncertainty_response, responses_are_substantive, token_f1
from config import QASample, SamplerConfig, UNCERTAINTY_TOKENS, COVERAGE_SIM_THRESHOLD, MMLU_SUBJECTS

In [27]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
assert DEVICE == "cuda", "Switch to GPU runtime: Runtime > Change runtime type > T4 GPU"

Device: cuda


In [28]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [29]:
triviaqa_samples   = load_triviaqa(SEED, n_total=3000)
webq_samples       = load_webquestions(SEED)
mmlu_samples       = load_mmlu(SEED)

DATASETS = {
    "triviaqa": triviaqa_samples,
    "webq":     webq_samples,
    "mmlu":     mmlu_samples,
}

# Quick sanity check
for name, samples in DATASETS.items():
    cal  = len(filter_split(samples, "calibration"))
    val  = len(filter_split(samples, "validation"))
    test = len(filter_split(samples, "test"))
    print(f"{name}: cal={cal}, val={val}, test={test}")

TriviaQA loaded: 3000 samples
  calibration: 1500
  validation: 750
  test: 750
WebQuestions loaded: 3778 samples
  calibration: 1889
  validation: 945
  test: 944
MMLU loaded: 2314 samples across 5 subjects
  calibration: 1157
  validation: 579
  test: 578
triviaqa: cal=1500, val=750, test=750
webq: cal=1889, val=945, test=944
mmlu: cal=1157, val=579, test=578


In [30]:
DEFAULT_CONFIG = SamplerConfig()

In [31]:
def adaptive_sample_fixed(
    question: str,
    config:   SamplerConfig = DEFAULT_CONFIG,
) -> dict:
    """
    ABLATION A: Remove adaptive stopping.

    Always samples the full budget (max_batches × batch_size = 18 responses).
    plateau_stop is never called. Uncertainty filter is kept.

    Everything else identical to adaptive_sample.
    """
    all_responses = []
    history       = []

    # Always run all batches — no plateau check
    for batch_num in range(1, config.max_batches + 1):
        new_responses = generate_batch(
            question,
            n=config.batch_size,
            temperature=config.temperature,
        )
        all_responses.extend(new_responses)

        stability = compute_stability(all_responses)
        history.append(stability)

    # After full budget, apply uncertainty filter
    cleaned = [clean_response(r) for r in all_responses]

    if not responses_are_substantive(cleaned):
        return {
            "abstain":   True,
            "reason":    "uncertainty_responses",
            "responses": all_responses,
            "cleaned":   cleaned,
            "stability": history[-1],
            "n_score":   None,
            "n_batches": config.max_batches,
            "n_samples": len(all_responses),
            "history":   history,
        }

    return {
        "abstain":   False,
        "reason":    "full_budget",
        "responses": all_responses,
        "cleaned":   cleaned,
        "stability": history[-1],
        "n_score":   round(1.0 - history[-1], 6),
        "n_batches": config.max_batches,
        "n_samples": len(all_responses),
        "history":   history,
    }

In [32]:
def adaptive_sample_no_filter(
    question: str,
    config:   SamplerConfig = DEFAULT_CONFIG,
) -> dict:
    """
    ABLATION B: Remove uncertainty filter.

    Adaptive stopping is kept. But when a plateau is reached on
    uncertainty responses ("unknown", "uncertain", etc.), a prediction
    set is returned anyway instead of forcing abstention.

    Only abstains on budget_exhausted (no plateau reached).
    """
    all_responses = []
    history       = []

    for batch_num in range(1, config.max_batches + 1):
        new_responses = generate_batch(
            question,
            n=config.batch_size,
            temperature=config.temperature,
        )
        all_responses.extend(new_responses)

        stability = compute_stability(all_responses)
        history.append(stability)

        if batch_num >= config.min_batches:
            if plateau_stop(history, eps=config.eps):
                cleaned = [clean_response(r) for r in all_responses]

                # NO uncertainty filter — return set regardless of content
                return {
                    "abstain":   False,
                    "reason":    "plateau",
                    "responses": all_responses,
                    "cleaned":   cleaned,
                    "stability": stability,
                    "n_score":   round(1.0 - stability, 6),
                    "n_batches": batch_num,
                    "n_samples": len(all_responses),
                    "history":   history,
                }

    # Budget exhausted — still abstain (no plateau means genuine instability)
    cleaned = [clean_response(r) for r in all_responses]
    return {
        "abstain":   True,
        "reason":    "budget_exhausted",
        "responses": all_responses,
        "cleaned":   cleaned,
        "stability": history[-1] if history else 0.0,
        "n_score":   None,
        "n_batches": config.max_batches,
        "n_samples": len(all_responses),
        "history":   history,
    }

In [33]:
def evaluate_ablation(
    dataset_name:   str,
    alpha:          float,
    sampler_fn,                          # which adaptive_sample variant to use
    cal_samples:    int   = 500,
    test_samples:   int   = 500,
    config:         SamplerConfig = DEFAULT_CONFIG,
) -> dict:
    """
    Calibrate using sampler_fn on calibration split,
    evaluate on test split. Returns ECR, SSC, APSS, API, abstain_rate.
    """
    import random

    # --- Calibration ---
    cal = filter_split(DATASETS[dataset_name], "calibration")
    if len(cal) > cal_samples:
        cal = random.sample(cal, cal_samples)

    n_scores    = []
    n_abstained = 0

    for sample in cal:
        clear_embed_cache()
        result = sampler_fn(sample.question, config=config)
        if result["abstain"]:
            n_abstained += 1
        else:
            n_scores.append(result["n_score"])

    q_hat = compute_qhat(n_scores, alpha)
    cal_abstain_rate = n_abstained / len(cal)

    print(f"  Calibration: n={len(cal)}, n_scored={len(n_scores)}, "
          f"n_abstained={n_abstained}, q_hat={q_hat:.4f}")

    # --- Evaluation ---
    test = filter_split(DATASETS[dataset_name], "test")
    if len(test) > test_samples:
        test = random.sample(test, test_samples)

    n_covered     = 0
    n_ssc_covered = 0
    n_ssc_total   = 0
    total_set_size  = 0
    total_api_calls = 0

    for i, sample in enumerate(test):
        clear_embed_cache()
        result   = sampler_fn(sample.question, config=config)
        pred_set = build_prediction_set(result, q_hat)
        covered  = check_coverage(pred_set, sample.gold_answers)
        abstained = result["abstain"] or (
            result["n_score"] is not None and result["n_score"] > q_hat
        )

        if covered:
            n_covered += 1
        if not abstained:
            n_ssc_total += 1
            if covered:
                n_ssc_covered += 1

        total_set_size  += len(pred_set)
        total_api_calls += result["n_samples"]

        if (i + 1) % 100 == 0:
            print(f"  test [{i+1}/{len(test)}] ECR={n_covered/(i+1)*100:.1f}%")

    n = len(test)
    return {
        "ECR":          round(n_covered / n * 100, 1),
        "SSC":          round((n_ssc_covered / n_ssc_total * 100)
                              if n_ssc_total > 0 else 0.0, 1),
        "APSS":         round(total_set_size / n, 2),
        "API":          round(total_api_calls / n, 1),
        "abstain_rate": round((n - n_ssc_total) / n * 100, 1),
        "q_hat":        q_hat,
    }

In [34]:
def run_ablations(alpha: float = ALPHA):

    datasets = ["triviaqa"]
    results  = {}

    for dataset in datasets:
        print(f"\n{'='*60}")
        print(f"ABLATION A (no adaptive stopping) — {dataset.upper()}")
        print(f"{'='*60}")
        r_fixed = evaluate_ablation(
            dataset_name = dataset,
            alpha        = alpha,
            sampler_fn   = adaptive_sample_fixed,
        )
        print(f"  → ECR={r_fixed['ECR']}  SSC={r_fixed['SSC']}  "
              f"APSS={r_fixed['APSS']}  API={r_fixed['API']}  "
              f"Abstain={r_fixed['abstain_rate']}%")

        print(f"\n{'='*60}")
        print(f"ABLATION B (no uncertainty filter) — {dataset.upper()}")
        print(f"{'='*60}")
        r_nofilter = evaluate_ablation(
            dataset_name = dataset,
            alpha        = alpha,
            sampler_fn   = adaptive_sample_no_filter,
        )
        print(f"  → ECR={r_nofilter['ECR']}  SSC={r_nofilter['SSC']}  "
              f"APSS={r_nofilter['APSS']}  API={r_nofilter['API']}  "
              f"Abstain={r_nofilter['abstain_rate']}%")

        results[dataset] = {
            "full":      FULL_ADAPTIVE[dataset],
            "no_stop":   r_fixed,
            "no_filter": r_nofilter,
        }

    # Print summary table
    print(f"\n\n{'='*72}")
    print(f"ABLATION SUMMARY  (alpha={alpha})")
    print(f"{'='*72}")

    for dataset in datasets:
        r = results[dataset]
        print(f"\n{dataset.upper()}")
        print(f"  {'Variant':<30} {'ECR':>6} {'SSC':>6} {'APSS':>6} "
              f"{'API':>5} {'Abstain':>8}")
        print(f"  {'-'*60}")

        variants = [
            ("Full AdaptiveCP",         r["full"]),
            ("A: No adaptive stopping", r["no_stop"]),
            ("B: No uncertainty filter",r["no_filter"]),
        ]

        for name, v in variants:
            ecr_ok = "✓" if v["ECR"] >= (1 - alpha) * 100 else "✗"
            print(f"  {name:<30} {v['ECR']:>5.1f}{ecr_ok} "
                  f"{v['SSC']:>6.1f} {v['APSS']:>6.2f} "
                  f"{v['API']:>5.1f} {v['abstain_rate']:>7.1f}%")

        # Print deltas vs full
        print(f"\n  Deltas vs Full AdaptiveCP:")
        full = r["full"]
        for name, v in variants[1:]:
            d_ecr  = v["ECR"]  - full["ECR"]
            d_ssc  = v["SSC"]  - full["SSC"]
            d_apss = v["APSS"] - full["APSS"]
            d_api  = v["API"]  - full["API"]
            print(f"  {name:<30} "
                  f"ECR{d_ecr:+.1f}  SSC{d_ssc:+.1f}  "
                  f"APSS{d_apss:+.2f}  API{d_api:+.1f}")

    # Save
    import json
    save = {ds: {k: v for k, v in r.items() if k != "results"}
            for ds, r in results.items()}
    with open("/content/ablation_results.json", "w") as f:
        json.dump(save, f, indent=2)
    print(f"\nSaved to /content/ablation_results.json")

    return results


ablation_results = run_ablations(alpha=ALPHA)



ABLATION A (no adaptive stopping) — TRIVIAQA
  Calibration: n=500, n_scored=500, n_abstained=0, q_hat=0.1015
  test [100/500] ECR=88.0%
  test [200/500] ECR=90.0%
  test [300/500] ECR=90.7%
  test [400/500] ECR=90.5%
  test [500/500] ECR=89.2%
  → ECR=89.2  SSC=83.7  APSS=0.93  API=18.0  Abstain=33.6%

ABLATION B (no uncertainty filter) — TRIVIAQA
  Calibration: n=500, n_scored=444, n_abstained=56, q_hat=0.0404
  test [100/500] ECR=90.0%
  test [200/500] ECR=91.5%
  test [300/500] ECR=92.0%
  test [400/500] ECR=92.2%
  test [500/500] ECR=92.0%
  → ECR=92.0  SSC=85.9  APSS=0.63  API=11.6  Abstain=43.4%


ABLATION SUMMARY  (alpha=0.35)

TRIVIAQA
  Variant                           ECR    SSC   APSS   API  Abstain
  ------------------------------------------------------------


KeyError: 'abstain_rate'